<a href="https://colab.research.google.com/github/iidakorhonen-ship-it/BUS4_118S/blob/main/Code_Generation_with_ReACT_Promptin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import ast
import sys
import traceback
from typing import Dict, List, Tuple, Any
import json

class ReACTCodeGenerator:
    def __init__(self):
        self.reasoning_history = []
        self.action_history = []
        self.observation_history = []

    def generate_code_with_react(self, task_description: str) -> Dict:
        """Main ReACT loop for code generation"""

        reasoning = self._reason_about_task(task_description)
        print(f"🧠 REASONING:\n{reasoning}\n")

        action_plan = self._create_action_plan(task_description, reasoning)
        print(f"📋 ACTION PLAN:\n{action_plan}\n")

        generated_code = self._generate_code(task_description, reasoning, action_plan)
        print(f"💻 GENERATED CODE:\n{generated_code}\n")

        execution_result = self._execute_and_observe(generated_code)
        print(f"👀 OBSERVATION:\n{execution_result}\n")

        if not execution_result['success']:
            improved_code = self._reflect_and_improve(
                task_description, generated_code, execution_result
            )
            print(f"🔄 IMPROVED CODE:\n{improved_code}\n")
            execution_result = self._execute_and_observe(improved_code)
            generated_code = improved_code

        return {
            'reasoning': reasoning,
            'action_plan': action_plan,
            'final_code': generated_code,
            'execution_result': execution_result
        }

    def _reason_about_task(self, task: str) -> str:
        """Reasoning phase - understand the problem"""
        reasoning_prompt = f"""
        Task: {task}

        Think step by step about this programming task:

        1. What is the core problem to solve?
        2. What are the inputs and expected outputs?
        3. What Python concepts/libraries will I need?
        4. What are potential edge cases or challenges?
        5. What's the best approach to solve this?

        Provide clear reasoning for each point.
        """

        reasoning = f"""
        REASONING FOR TASK: {task}

        1. CORE PROBLEM: Need to create a function that processes data according to specifications
        2. INPUTS/OUTPUTS: Analyzing the requirements to determine data types and return values
        3. LIBRARIES NEEDED: Identifying which Python standard library or third-party packages to use
        4. EDGE CASES: Considering error handling, empty inputs, invalid data types
        5. APPROACH: Breaking down into smaller functions, using appropriate algorithms

        The task requires careful consideration of efficiency and readability.
        """

        self.reasoning_history.append(reasoning)
        return reasoning

    def _create_action_plan(self, task: str, reasoning: str) -> str:
        """Action planning phase"""
        action_prompt = f"""
        Task: {task}
        Reasoning: {reasoning}

        Based on the reasoning above, create a detailed action plan:

        1. Define the main function signature
        2. List helper functions needed
        3. Outline the algorithm steps
        4. Plan error handling
        5. Consider testing approach

        Make this a concrete implementation plan.
        """

        action_plan = """
        ACTION PLAN:

        1. MAIN FUNCTION: Define clear function with proper parameters and return type
        2. HELPER FUNCTIONS: Create utility functions for data validation and processing
        3. ALGORITHM STEPS:
           - Validate input parameters
           - Process data according to requirements
           - Handle edge cases
           - Return formatted results
        4. ERROR HANDLING: Use try-catch blocks and input validation
        5. TESTING: Include example usage and test cases
        """

        self.action_history.append(action_plan)
        return action_plan

    def _generate_code(self, task: str, reasoning: str, plan: str) -> str:
        """Generate actual Python code"""
        code_prompt = f"""
        Task: {task}
        Reasoning: {reasoning}
        Plan: {plan}

        Generate clean, well-commented Python code that:
        1. Follows the action plan exactly
        2. Includes proper error handling
        3. Has clear variable names and structure
        4. Includes docstrings and comments
        5. Provides example usage

        Return ONLY the Python code, no additional text.
        """

        generated_code = '''
def process_data(data_list, operation="sum", filter_condition=None):
    """
    Process a list of numbers with specified operation and optional filtering.

    Args:
        data_list (list): List of numbers to process
        operation (str): Operation to perform ('sum', 'avg', 'max', 'min')
        filter_condition (callable, optional): Function to filter data

    Returns:
        float: Result of the operation

    Raises:
        ValueError: If data_list is empty or operation is invalid
        TypeError: If data_list contains non-numeric values
    """

    # Validate inputs
    if not data_list:
        raise ValueError("data_list cannot be empty")

    if not isinstance(data_list, list):
        raise TypeError("data_list must be a list")

    # Filter data if condition provided
    if filter_condition:
        try:
            filtered_data = [x for x in data_list if filter_condition(x)]
        except Exception as e:
            raise ValueError(f"Filter condition error: {e}")
    else:
        filtered_data = data_list

    # Validate numeric data
    try:
        numeric_data = [float(x) for x in filtered_data]
    except (ValueError, TypeError) as e:
        raise TypeError(f"All items must be numeric: {e}")

    if not numeric_data:
        raise ValueError("No data remains after filtering")

    # Perform operation
    operations = {
        'sum': sum,
        'avg': lambda x: sum(x) / len(x),
        'max': max,
        'min': min
    }

    if operation not in operations:
        raise ValueError(f"Invalid operation. Must be one of: {list(operations.keys())}")

    return operations[operation](numeric_data)

# Example usage
if __name__ == "__main__":
    # Test cases
    test_data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

    print("Sum:", process_data(test_data, "sum"))
    print("Average:", process_data(test_data, "avg"))
    print("Max:", process_data(test_data, "max"))
    print("Filtered sum (even numbers):", process_data(test_data, "sum", lambda x: x % 2 == 0))
'''

        return generated_code

    def _execute_and_observe(self, code: str) -> Dict:
        """Execute code and observe results"""
        try:
            exec_namespace = {}

            exec(code, exec_namespace)

            result = {
                'success': True,
                'output': 'Code executed successfully',
                'errors': None
            }

            self.observation_history.append(result)
            return result

        except Exception as e:
            error_result = {
                'success': False,
                'output': None,
                'errors': {
                    'type': type(e).__name__,
                    'message': str(e),
                    'traceback': traceback.format_exc()
                }
            }

            self.observation_history.append(error_result)
            return error_result

    def _reflect_and_improve(self, task: str, code: str, execution_result: Dict) -> str:
        """Reflect on errors and improve code"""
        reflection_prompt = f"""
        Original Task: {task}
        Generated Code: {code}
        Execution Error: {execution_result['errors']}

        The code failed to execute. Analyze the error and provide an improved version:

        1. What caused the error?
        2. How can it be fixed?
        3. What improvements can be made?
        4. Generate the corrected code.

        Focus on fixing the specific error while maintaining the original functionality.
        """

        improved_code = code.replace("# Example usage", """
# Example usage - Fixed version
def safe_test():
    try:
        test_data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

        print("Sum:", process_data(test_data, "sum"))
        print("Average:", process_data(test_data, "avg"))
        print("Max:", process_data(test_data, "max"))
        print("Filtered sum (even numbers):", process_data(test_data, "sum", lambda x: x % 2 == 0))
        return True
    except Exception as e:
        print(f"Error in test: {e}")
        return False

# Example usage""")

        return improved_code

def demonstrate_react_coding():
    generator = ReACTCodeGenerator()

    task = "Create a function that processes a list of numbers with different operations (sum, average, max, min) and optional filtering"

    print("=== ReACT CODE GENERATION DEMO ===\n")

    result = generator.generate_code_with_react(task)

    print("=== FINAL RESULT ===")
    print(f"Success: {result['execution_result']['success']}")
    if result['execution_result']['errors']:
        print(f"Errors: {result['execution_result']['errors']}")

if __name__ == "__main__":
    demonstrate_react_coding()


=== ReACT CODE GENERATION DEMO ===

🧠 REASONING:

        REASONING FOR TASK: Create a function that processes a list of numbers with different operations (sum, average, max, min) and optional filtering
        
        1. CORE PROBLEM: Need to create a function that processes data according to specifications
        2. INPUTS/OUTPUTS: Analyzing the requirements to determine data types and return values
        3. LIBRARIES NEEDED: Identifying which Python standard library or third-party packages to use
        4. EDGE CASES: Considering error handling, empty inputs, invalid data types
        5. APPROACH: Breaking down into smaller functions, using appropriate algorithms
        
        The task requires careful consideration of efficiency and readability.
        

📋 ACTION PLAN:

        ACTION PLAN:
        
        1. MAIN FUNCTION: Define clear function with proper parameters and return type
        2. HELPER FUNCTIONS: Create utility functions for data validation and processing